# Formativa 3 · Eficiencia y complejidad del método IQR

**Magíster:** Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Programación para la Ciencia de Datos  
**Código:** MCDI500  
**Docente:** Omar Salinas  
**Universidad:** Universidad Andrés Bello  
**Grupo:** 7  

**Integrantes:**
- Benjamín Araya Matta
- Alan Cañete Calquin
- Patricio Cortez Triviño
- Jorge Guaico Ortega

## Objetivo

Analizar la eficiencia de dos implementaciones del método del rango intercuartílico (IQR) para detectar valores atípicos en la variable `MontoNetoOC_CLP` del conjunto de datos trabajado durante la Fase 2.

## Alcance del trabajo

Se desarrollará una implementación mediante un bucle y otra vectorizada con pandas. Ambas deberán producir el mismo resultado. Posteriormente se compararán sus tiempos de ejecución, uso de memoria y complejidad computacional. También se justificará si la recursividad es necesaria para resolver este problema.

## 1. Fundamento del método IQR

El rango intercuartílico (IQR) es una medida de dispersión que representa la distancia entre el primer cuartil (Q1) y el tercer cuartil (Q3):

**IQR = Q3 - Q1**

Para detectar posibles valores atípicos se calculan los siguientes límites:

- **Límite inferior:** Q1 - 1.5 × IQR
- **Límite superior:** Q3 + 1.5 × IQR

Se considera como posible valor atípico cualquier dato que se encuentre fuera de estos límites. En este trabajo, el método se aplicará a la variable `MontoNetoOC_CLP`, correspondiente al monto neto de las órdenes de compra.

## 2. Carga y selección de los datos

Se utilizará el archivo procesado `ordenes.csv`, generado durante la Fase 2. Para este análisis se cargará la variable `MontoNetoOC_CLP`, que conserva los montos expresados en pesos chilenos después del proceso de limpieza.

Aunque el archivo incluye una clasificación previa de valores extremos, el método IQR será implementado nuevamente mediante dos estrategias —un bucle y una operación vectorizada— para comparar sus resultados y eficiencia.

In [1]:
from pathlib import Path
import pandas as pd

# Buscar automáticamente la carpeta principal del proyecto
inicio = Path.cwd().resolve()
candidatos = [inicio, *inicio.parents]

ROOT = next(
    (
        carpeta
        for carpeta in candidatos
        if (carpeta / "F2" / "data" / "processed" / "ordenes.csv").exists()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError("No se encontró el archivo procesado ordenes.csv")

# Cargar solamente la columna que utilizaremos
ruta_datos = ROOT / "F2" / "data" / "processed" / "ordenes.csv"

datos = pd.read_csv(
    ruta_datos,
    usecols=["MontoNetoOC_CLP"]
)

datos.head()

,MontoNetoOC_CLP
0,515000.0
1,1764706.0
2,526400.0
3,1035433.0
4,5638800.0


In [2]:
# Validar la columna antes de calcular el IQR
datos["MontoNetoOC_CLP"] = pd.to_numeric(
    datos["MontoNetoOC_CLP"],
    errors="coerce"
)

resumen_carga = {
    "filas_totales": len(datos),
    "valores_validos": datos["MontoNetoOC_CLP"].notna().sum(),
    "valores_faltantes": datos["MontoNetoOC_CLP"].isna().sum()
}

resumen_carga

{'filas_totales': 541,
 'valores_validos': np.int64(541),
 'valores_faltantes': np.int64(0)}

In [3]:
# Crear una serie limpia con los montos que analizaremos
montos = datos["MontoNetoOC_CLP"].dropna().reset_index(drop=True)

# Revisar sus principales estadísticas descriptivas
montos.describe()

count    5.410000e+02
mean     1.236630e+06
std      1.526727e+06
min      1.727500e+04
25%      2.179230e+05
50%      5.264000e+05
75%      1.620152e+06
max      6.795000e+06
Name: MontoNetoOC_CLP, dtype: float64

In [4]:
# Calcular los cuartiles y los límites del método IQR
q1 = montos.quantile(0.25)
q3 = montos.quantile(0.75)

iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

resultados_iqr = {
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
    "limite_inferior": limite_inferior,
    "limite_superior": limite_superior
}

resultados_iqr

{'Q1': np.float64(217923.0),
 'Q3': np.float64(1620152.0),
 'IQR': np.float64(1402229.0),
 'limite_inferior': np.float64(-1885420.5),
 'limite_superior': np.float64(3723495.5)}

### 3. Cálculo de los límites del método IQR

A partir de los 541 montos válidos se obtuvo un primer cuartil (Q1) de 217.923 CLP y un tercer cuartil (Q3) de 1.620.152 CLP. Por lo tanto, el rango intercuartílico es de 1.402.229 CLP.

La aplicación del criterio de 1,5 veces el IQR establece un límite inferior de -1.885.420,50 CLP y un límite superior de 3.723.495,50 CLP. Debido a que los montos analizados son positivos, los posibles valores atípicos se concentrarán sobre el límite superior.

Estos mismos límites serán utilizados en las implementaciones con bucle y vectorizada, permitiendo comparar ambos procedimientos bajo condiciones equivalentes.